# 丢弃层

上一章我们用展平层 + 两个线性层搭建了第一个图像识别模型，在 TinyMNIST 上达到了 92% 的准确率。

这很不错，但有一个潜在的隐患：**过拟合**（Overfitting）。

什么是过拟合？想象你正在复习考试，但只反复研究了历年真题，把每道题的答案都背了下来。一旦遇到稍有不同的新题，就完全不知道怎么下手。你记住的是答案，而不是解题方法。

神经网络也会犯同样的毛病，学习了过多的细节。如果训练数据里的猫都有清晰的条纹，模型可能会学到 **有条纹 = 猫** 这个规律，而不是从整体形状来判断。换一批猫的照片，准确率就会大幅下降。

过拟合的模型在训练集上表现很好，但在测试集（从未见过的数据）上表现就会很差。

---

为了解决过拟合，“深度学习三巨头”之一的辛顿（Hinton）和他的团队提出了一个简单却极具智慧的方法：**丢弃层**（Dropout Layer）。

丢弃层的核心思想是：**训练时，随机将一部分神经元的输出强制置零**。每次迭代，被置零的神经元位置都不同（随机掩码）。

这就好比在照片里随机添加一些白噪音，让网络模型无法再发现过多的细节，而是更加关注整体的轮廓和结构。

```{figure} images/dropout.png
:align: center
:width: 480px
**图例：丢弃层示意图**
```

In [1]:
from abc import abstractmethod, ABC

import numpy as np

np.random.seed(99)

## 基础架构

### 张量

In [2]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for p in self.parents:
            p.backward()

    @property
    def shape(self):
        return self.data.shape

    def __repr__(self):
        return f'Tensor({self.data})'

### 基础数据集

In [3]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size

        self.test_features = None
        self.test_labels = None
        self.train_features = None
        self.train_labels = None

        self.load()

        self.features = self.train_features
        self.labels = self.train_labels

    @abstractmethod
    def load(self):
        pass

    def train(self):
        self.features = self.train_features
        self.labels = self.train_labels

    def eval(self):
        self.features = self.test_features
        self.labels = self.test_labels

    def items(self):
        return Tensor(self.features), Tensor(self.labels)

    def __len__(self):
        return len(self.features) // self.batch_size

    def __getitem__(self, index):
        start = index * self.batch_size
        end = start + self.batch_size

        feature = Tensor(self.features[start: end])
        label = Tensor(self.labels[start: end])
        return feature, label

### 基础层

我们在基础层里加入两个模式切换函数：

* `train()`：切换到训练模式；
* `eval()`：切换到测试模式。

为什么需要区分两种模式？因为丢弃层只在训练时随机置零，测试时必须让所有神经元正常工作，才能得到稳定的预测结果。通过 `self.training` 这个标志位，每个层都可以根据当前模式自行决定该如何运作。

In [4]:
class Layer(ABC):

    def __init__(self):
        self.training = True

    def __call__(self, x: Tensor):
        return self.forward(x)

    def train(self):
        self.training = True

    def eval(self):
        self.training = False

    @abstractmethod
    def forward(self, x: Tensor):
        pass

    @property
    def parameters(self):
        return []

    def __repr__(self):
        return f"{type(self).__name__}[]"

### 基础损失函数

In [5]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor):
        pass

### 基础优化器

In [6]:
class Optimizer(ABC):

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def reset(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

    @abstractmethod
    def step(self):
        pass

### 基础模型

In [7]:
class Model(ABC):

    def __init__(self, layer, loss_fn, optimizer):
        self.layer = layer
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    @abstractmethod
    def train(self, dataset, epochs):
        pass

    @abstractmethod
    def test(self, dataset):
        pass

## 数据

### MNIST 数据集

In [8]:
class MNISTDataset(Dataset):

    def __init__(self, filename, batch_size=1):
        self.filename = filename
        super().__init__(batch_size)

    def load(self):
        with np.load(self.filename, allow_pickle=True) as f:
            self.train_features, self.train_labels = self.preprocess(f['x_train'], f['y_train'])
            self.test_features, self.test_labels = self.preprocess(f['x_test'], f['y_test'])

    @staticmethod
    def preprocess(x, y):
        inputs = x / 255.0
        inputs = np.expand_dims(inputs, axis=1)
        targets = np.zeros((len(y), 10))
        targets[np.arange(len(y)), y] = 1
        return inputs, targets

    def estimate(self, predictions):
        predicted_digits = predictions.data.argmax(axis=1)
        digits = self.labels.argmax(axis=1)
        correct = (predicted_digits == digits).sum()
        return correct / len(self.labels)

## 模型

### 线性层

In [9]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        super().__init__()
        self.weight = Tensor(np.random.rand(out_size, in_size) / in_size)
        self.bias = Tensor(np.random.rand(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad += np.sum(p.grad, axis=0)
            x.grad += p.grad @ self.weight.data

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

    def __repr__(self):
        return f'Linear[weight{self.weight.shape}; bias{self.bias.shape}]'

### 顺序层

有了层的模式切换机制，顺序层也需要相应更新：调用 `train()` 或 `eval()` 时，要把信号传递给所有子层，确保每一层都切换到正确的模式。

In [10]:
class Sequential(Layer):

    def __init__(self, layers):
        super().__init__()
        self.layers = layers

    def train(self):
        for l in self.layers:
            l.train()

    def eval(self):
        for l in self.layers:
            l.eval()

    def forward(self, x: Tensor):
        for l in self.layers:
            x = l(x)
        return x

    @property
    def parameters(self):
        return [p for l in self.layers for p in l.parameters]

    def __repr__(self):
        return '\n'.join(str(l) for l in self.layers if str(l))

### 展平层

In [11]:
class Flatten(Layer):

    def forward(self, x: Tensor):
        p = Tensor(np.array(x.data.reshape(x.data.shape[0], -1)))

        def gradient_fn():
            x.grad += p.grad.reshape(x.data.shape)

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

### 丢弃层

**前向传播**：

* 测试模式：直接返回输入，不做任何处理；
* 训练模式：生成一个与输入形状相同的**随机掩码**（mask），并以此来控制输入数据是否向下一层传递。

**梯度计算**：

反向传播时，必须使用**和前向传播完全相同的掩码**。道理很简单：前向传播中某个神经元被置零了，那它对输出没有任何贡献，梯度也应该是 0，不应该往前继续传。用同一份掩码遮住梯度，就能保证梯度流向和前向传播完全一致。

为此，掩码需要在 `forward()` 里生成，并通过闭包被 `gradient_fn()` 捕获，在反向传播时复用。

**参数**：丢弃层没有可学习的参数，`parameters` 为空。

**丢弃率**（`dropout_rate`）是一个**超参数**，控制随机丢弃的比例。默认值 0.2 表示每次训练随机丢弃 20% 的神经元。

In [12]:
class Dropout(Layer):

    def __init__(self, dropout_rate=0.2):
        super().__init__()
        self.dropout_rate = dropout_rate

    def forward(self, x: Tensor):
        if not self.training:
            return x

        mask = np.random.random(x.data.shape) > self.dropout_rate
        p = Tensor(x.data * mask)

        def gradient_fn():
            x.grad += p.grad * mask

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    def __repr__(self):
        return f'Dropout[rate={self.dropout_rate}]'

### ReLU 激活函数层

In [13]:
class ReLU(Layer):

    def forward(self, x: Tensor):
        a = Tensor(np.maximum(0, x.data))

        def gradient_fn():
            x.grad += a.grad * (a.data > 0)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

### 均方误差损失函数

In [14]:
class MSELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            p.grad += -2 * (y.data - p.data) / len(y.data)

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

### 随机梯度下降优化器

In [15]:
class SGDOptimizer(Optimizer):

    def step(self):
        for p in self.parameters:
            p.data -= p.grad * self.lr

### 神经网络模型

模型的 `train()` 和 `test()` 方法需要同步切换两件事：
* **数据集模式**：`dataset.train()` 切换到训练集，`dataset.eval()` 切换到测试集；
* **层的模式**：`self.layer.train()` 或 `self.layer.eval()`，通知丢弃层等是否激活。

两者都要切换，缺一不可。

In [16]:
class NNModel(Model):

    def train(self, dataset, epochs):
        self.layer.train()
        dataset.train()

        for epoch in range(epochs):
            for i in range(len(dataset)):
                features, labels = dataset[i]

                self.optimizer.reset()
                predictions = self.layer(features)
                loss = self.loss_fn(predictions, labels)
                loss.backward()
                self.optimizer.step()

    def test(self, dataset):
        self.layer.eval()
        dataset.eval()

        features, labels = dataset.items()
        predictions = self.layer(features)
        loss = self.loss_fn(predictions, labels)
        return predictions, loss

## 训练

### 超参数：学习率

In [17]:
LEARNING_RATE = 0.01

### 超参数：批大小

In [18]:
BATCH_SIZE = 2

### 超参数：轮次

In [19]:
EPOCHS = 10

### 建模

丢弃层通常放在参数量较大的全连接线性层之后，这里是发生过拟合风险最高的位置。现在我们把它放在隐藏层的 ReLU 后面。

丢弃率使用默认值 0.2，即每次训练随机丢弃 20% 的神经元。

In [20]:
dataset = MNISTDataset('tinymnist.npz', BATCH_SIZE)

layer = Sequential([Flatten(),
                    Linear(1 * 28 * 28, 64),
                    ReLU(),
                    Dropout(),
                    Linear(64, 10)])
loss_fn = MSELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)

model = NNModel(layer, loss_fn, optimizer)
print(layer)

Flatten[]
Linear[weight(64, 784); bias(64,)]
ReLU[]
Dropout[rate=0.2]
Linear[weight(10, 64); bias(10,)]


### 迭代

In [21]:
model.train(dataset, EPOCHS)

## 验证

### 测试

调用 `model.test()` 会自动切换到测试模式，丢弃层此时直接透传输入，不做任何丢弃操作。

In [22]:
predictions, loss = model.test(dataset)
accuracy = dataset.estimate(predictions)
print(f'accuracy: {accuracy:.2%}')

accuracy: 90.90%


加入丢弃层后，准确率从 92.0% 略降到约 90.9%。这其实是正常现象，原因有两个：

首先，丢弃层是为了提高模型对**新数据的泛化能力**，会让训练过程变得更难，因为网络每次都在「残缺」的状态下学习。相同轮次内，它可能不如没有丢弃层的模型收敛得快，但在更大、更复杂的数据集上，最终往往能取得更好的效果。

其次，MNIST 是一个相对“干净”的数据集，本身过拟合风险不高，丢弃层在这里的收益有限。在更复杂的真实场景（比如自然图片分类），丢弃层的效果会更加显著。

## 课后练习

把 `DROPOUT_RATE = 0.2` 设置为超参数，分别尝试 0.1、0.3、0.5，观察不同丢弃率对训练准确率的影响。丢弃率越大，一定越好吗？